In [1]:
# =============================================================================
# Ontario Wildfire Cause Frequency Map — 2010–2022
# Pixels coloured by dominant fire cause, intensity by fire count
# Human=red, Natural=green, Undetermined=yellow, Prescribed=grey
# No-fire pixels=light blue
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import glob
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('figures', exist_ok=True)

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR       = r'C:\Users\lambe\RU_Thesis_2026\ml_data'
TRAIN_YEARS    = list(range(2010, 2021))
VAL_YEARS      = [2021, 2022]
ALL_YEARS      = TRAIN_YEARS + VAL_YEARS
DOY_START_HARD = 91
DOY_END_HARD   = 310
COORD_ROUND    = 2   # round to ~9km pixel grid

# Cause codes
CAUSE_LABELS = {
    1: 'Undetermined',
    2: 'Human',
    3: 'Natural (lightning)',
    4: 'Prescribed burn',
}

# Base colours per cause (will be darkened by fire count)
BASE_COLORS = {
    1: np.array([1.00, 0.95, 0.00]),   # yellow  — undetermined
    2: np.array([0.85, 0.10, 0.10]),   # red     — human
    3: np.array([0.10, 0.65, 0.10]),   # green   — natural
    4: np.array([0.55, 0.55, 0.55]),   # grey    — prescribed
}
NO_FIRE_COLOR = '#a8d8ea'              # light blue — no fire

# =============================================================================
# STEP 1: Load all fire pixels from training + validation data
# =============================================================================
print("Loading fire pixel data (2010–2022)...")

KEEP_COLS = ['date', 'longitude', 'latitude',
             'target_ignition', 'fire_cause']

all_dfs = []
for year in ALL_YEARS:
    split = 'train' if year in TRAIN_YEARS else 'val'
    files = sorted(glob.glob(
        os.path.join(DATA_DIR, f'ML_{split}_{year}??.csv')))
    year_dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, usecols=KEEP_COLS, on_bad_lines='skip')
            year_dfs.append(df)
        except Exception as e:
            print(f"  WARNING: {os.path.basename(f)}: {e}")
    if year_dfs:
        df_year = pd.concat(year_dfs, ignore_index=True)
        n_fire  = (df_year['target_ignition'] == 1).sum()
        print(f"  {year}: {len(df_year):>10,} rows  ({n_fire:,} fires)")
        all_dfs.append(df_year)

df_all = pd.concat(all_dfs, ignore_index=True)

# ── Filter to fire season ──────────────────────────────────────────────────────
df_all['date'] = df_all['date'].astype(int).astype(str)
df_all['doy']  = pd.to_datetime(
    df_all['date'], format='%Y%m%d', errors='coerce').dt.dayofyear
df_all = df_all[(df_all['doy'] >= DOY_START_HARD) &
                (df_all['doy'] <= DOY_END_HARD)].drop(columns='doy')
df_all['target_ignition'] = df_all['target_ignition'].astype(int)
df_all['fire_cause']      = df_all['fire_cause'].astype(int)

# Round coordinates to pixel grid
df_all['lon_r'] = df_all['longitude'].round(COORD_ROUND)
df_all['lat_r'] = df_all['latitude'].round(COORD_ROUND)

print(f"\n  Total rows (fire season): {len(df_all):,}")
print(f"  Total fire pixels       : {(df_all['target_ignition']==1).sum():,}")

# ── Get unique pixel locations for no-fire background ─────────────────────────
print("\nExtracting unique pixel locations...")
df_pixels = df_all.drop_duplicates(
    subset=['lon_r', 'lat_r'])[['lon_r', 'lat_r']].copy()
print(f"  Unique pixels: {len(df_pixels):,}")

# =============================================================================
# STEP 2: Aggregate fire events per pixel
# =============================================================================
print("\nAggregating fire events per pixel...")

df_fires = df_all[df_all['target_ignition'] == 1].copy()

# Per-pixel cause counts
cause_counts = df_fires.groupby(
    ['lon_r', 'lat_r', 'fire_cause']).size().reset_index(name='count')

# Per-pixel total fire count and dominant cause
pixel_stats = (
    df_fires
    .groupby(['lon_r', 'lat_r'])
    .agg(
        total_fires = ('target_ignition', 'count'),
        dominant_cause = ('fire_cause',
                          lambda x: x.value_counts().index[0])
    )
    .reset_index()
)

print(f"  Pixels with at least one fire: {len(pixel_stats):,}")
print(f"\n  Fire count distribution:")
print(f"    Max fires in one pixel: {pixel_stats['total_fires'].max()}")
print(f"    Mean fires per fire pixel: "
      f"{pixel_stats['total_fires'].mean():.2f}")

print(f"\n  Dominant cause distribution:")
for code, label in CAUSE_LABELS.items():
    n = (pixel_stats['dominant_cause'] == code).sum()
    if n > 0:
        print(f"    {label:<25}: {n:,} pixels")

# =============================================================================
# STEP 3: Compute colour per pixel
# =============================================================================
print("\nComputing pixel colours...")

def compute_pixel_color(row, max_fires, min_scale=0.25):
    """
    Returns an RGB colour for a fire pixel.
    Base colour determined by dominant cause.
    Darker = more fires (intensity scales with log fire count).
    min_scale: minimum brightness multiplier (0=black, 1=base colour)
    """
    cause     = int(row['dominant_cause'])
    n_fires   = int(row['total_fires'])
    base_col  = BASE_COLORS.get(cause, np.array([0.5, 0.5, 0.5]))

    # Log scale: 1 fire = min_scale brightness, max_fires = full darkness
    if max_fires <= 1:
        scale = 1.0
    else:
        log_n   = np.log1p(n_fires)
        log_max = np.log1p(max_fires)
        # scale goes from 1.0 (1 fire) down to min_scale (max fires)
        scale   = 1.0 - (1.0 - min_scale) * (log_n / log_max)

    return tuple(np.clip(base_col * scale, 0, 1))

max_fires = pixel_stats['total_fires'].max()

pixel_stats['color'] = pixel_stats.apply(
    compute_pixel_color, axis=1, max_fires=max_fires)

print(f"  Colours computed for {len(pixel_stats):,} fire pixels")

# =============================================================================
# STEP 4: Compute dot size
# =============================================================================
lon_range = df_pixels['lon_r'].max() - df_pixels['lon_r'].min()
lat_range = df_pixels['lat_r'].max() - df_pixels['lat_r'].min()

FIG_W, FIG_H   = 16, 12
DPI            = 300
AXES_W         = FIG_W * 0.80
AXES_H         = FIG_H * 0.78
PIXEL_STEP     = 0.09   # ~9 km

PX_PER_DEG_X  = (AXES_W * DPI) / lon_range
PX_PER_DEG_Y  = (AXES_H * DPI) / lat_range
PIXEL_PX      = PIXEL_STEP * min(PX_PER_DEG_X, PX_PER_DEG_Y)
POINTS_PER_PX = DPI / 72
S_BASE        = (PIXEL_PX / POINTS_PER_PX) ** 2 * 0.72

print(f"\n  Dot size (s): {S_BASE:.1f}")

# =============================================================================
# STEP 5: Plot
# =============================================================================
print("\nGenerating cause frequency map...")

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
fig.patch.set_facecolor('white')

# ── Layer 1: No-fire pixels (background) ──────────────────────────────────────
# Merge pixel grid with fire pixels to find no-fire locations
fire_pixel_set = set(
    zip(pixel_stats['lon_r'], pixel_stats['lat_r']))
df_nofire = df_pixels[
    ~df_pixels.apply(
        lambda r: (r['lon_r'], r['lat_r']) in fire_pixel_set,
        axis=1)
].copy()

ax.scatter(
    df_nofire['lon_r'], df_nofire['lat_r'],
    c          = NO_FIRE_COLOR,
    s          = S_BASE * 0.85,
    alpha      = 0.6,
    linewidths = 0,
    rasterized = True,
    zorder     = 2,
    marker     = 's'
)
print(f"  No-fire pixels plotted: {len(df_nofire):,}")

# ── Layer 2: Fire pixels coloured by cause and count ──────────────────────────
ax.scatter(
    pixel_stats['lon_r'],
    pixel_stats['lat_r'],
    c          = list(pixel_stats['color']),
    s          = S_BASE * 1.2,
    alpha      = 0.95,
    linewidths = 0,
    rasterized = True,
    zorder     = 3,
    marker     = 's'
)
print(f"  Fire pixels plotted    : {len(pixel_stats):,}")

# ── Axis limits ────────────────────────────────────────────────────────────────
ax.set_xlim(df_pixels['lon_r'].min() - 0.5,
            df_pixels['lon_r'].max() + 0.5)
ax.set_ylim(df_pixels['lat_r'].min() - 0.3,
            df_pixels['lat_r'].max() + 0.3)

# ── Legend — cause colours ─────────────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(
        facecolor = NO_FIRE_COLOR,
        edgecolor = '#888888',
        linewidth = 0.5,
        label     = 'No fire recorded')
]

for code, label in CAUSE_LABELS.items():
    if (pixel_stats['dominant_cause'] == code).any():
        # Show light (1 fire) and dark (many fires) swatches
        base = BASE_COLORS[code]
        dark = tuple(np.clip(base * 0.25, 0, 1))
        legend_patches.append(mpatches.Patch(
            facecolor = tuple(base),
            edgecolor = 'none',
            label     = f'{label} (1 fire)'))
        legend_patches.append(mpatches.Patch(
            facecolor = dark,
            edgecolor = 'none',
            label     = f'{label} (many fires)'))

legend = ax.legend(
    handles        = legend_patches,
    title          = 'Dominant fire cause  '
                     '(darker = more fires, log scale)',
    title_fontsize = 9.5,
    fontsize       = 8.5,
    loc            = 'lower left',
    framealpha     = 0.95,
    edgecolor      = '#cccccc',
    ncol           = 2
)
legend.get_title().set_fontweight('bold')

# ── Colorbar — fire count intensity ───────────────────────────────────────────
# Show a generic grey-scale bar indicating fire count intensity
cbar_ax  = fig.add_axes([0.82, 0.25, 0.02, 0.45])
cmap_bar = plt.cm.Greys_r
norm_bar = mcolors.LogNorm(vmin=1, vmax=max_fires)
sm       = plt.cm.ScalarMappable(cmap=cmap_bar, norm=norm_bar)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Number of fire events\n(log scale)',
               fontsize=8.5, labelpad=8)
cbar.ax.tick_params(labelsize=8)

# ── Labels ─────────────────────────────────────────────────────────────────────
ax.set_xlabel('Longitude (°W)', fontsize=11)
ax.set_ylabel('Latitude (°N)',  fontsize=11)
ax.set_title(
    'Ontario Wildfire Ignition Frequency by Cause — 2010–2022\n'
    'Dominant cause per pixel  |  Colour intensity = fire count '
    '(log scale)',
    fontsize=13, fontweight='bold', pad=14
)
ax.tick_params(labelsize=10)
ax.grid(color='#e8e8e8', linewidth=0.4, alpha=0.6, zorder=1)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)

# ── Stats annotation ────────────────────────────────────────────────────────────
n_human  = (pixel_stats['dominant_cause'] == 2).sum()
n_nat    = (pixel_stats['dominant_cause'] == 3).sum()
n_undet  = (pixel_stats['dominant_cause'] == 1).sum()
n_presc  = (pixel_stats['dominant_cause'] == 4).sum()
n_total_fires = pixel_stats['total_fires'].sum()

stats_text = (
    f"Period         : 2010–2022\n"
    f"Fire pixels    : {len(pixel_stats):,}\n"
    f"Total events   : {n_total_fires:,}\n"
    f"──────────────────────────\n"
    f"Human          : {n_human:,} pixels\n"
    f"Natural        : {n_nat:,} pixels\n"
    f"Undetermined   : {n_undet:,} pixels\n"
    f"Prescribed     : {n_presc:,} pixels\n"
    f"──────────────────────────\n"
    f"Max fires/pixel: {max_fires:,}"
)
ax.text(
    0.985, 0.985, stats_text,
    transform           = ax.transAxes,
    fontsize            = 8.5,
    verticalalignment   = 'top',
    horizontalalignment = 'right',
    fontfamily          = 'monospace',
    bbox                = dict(boxstyle='round,pad=0.6',
                               facecolor='white',
                               edgecolor='#cccccc',
                               alpha=0.95)
)

fig.text(
    0.5, -0.005,
    'Source: Canadian National Fire Database (CNFDB)  |  '
    'Training: 2010–2020  |  Validation: 2021–2022  |  '
    'Fire season: DOY 91–310',
    ha='center', fontsize=8, color='#888888', style='italic'
)

plt.tight_layout()
fig.savefig('figures/fig_ontario_fire_cause_frequency_map.png',
            dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig('figures/fig_ontario_fire_cause_frequency_map.pdf',
            bbox_inches='tight', facecolor='white')

print("\n  Saved: figures/fig_ontario_fire_cause_frequency_map.png")
print("  Saved: figures/fig_ontario_fire_cause_frequency_map.pdf")
plt.close(fig)

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("CAUSE FREQUENCY MAP — SUMMARY")
print(f"{'='*55}")
print(f"  Period         : 2010–2022")
print(f"  Fire pixels    : {len(pixel_stats):,}")
print(f"  Total events   : {n_total_fires:,}")
print(f"  Max fires/pixel: {max_fires:,}")
print(f"\n  Dominant cause breakdown:")
for code, label in CAUSE_LABELS.items():
    n = (pixel_stats['dominant_cause'] == code).sum()
    pct = n / len(pixel_stats) * 100 if len(pixel_stats) > 0 else 0
    print(f"    {label:<25}: {n:,}  ({pct:.1f}%)")
print(f"\nLaTeX:")
print(f"  \\includegraphics[width=\\textwidth]"
      f"{{figures/fig_ontario_fire_cause_frequency_map}}")

Loading fire pixel data (2010–2022)...
  2010:  4,331,090 rows  (850 fires)
  2011:  4,331,090 rows  (1,263 fires)
  2012:  4,342,956 rows  (1,504 fires)
  2013:  4,331,090 rows  (532 fires)
  2014:  4,331,090 rows  (285 fires)
  2015:  4,331,090 rows  (634 fires)
  2016:  4,342,956 rows  (595 fires)
  2017:  4,331,090 rows  (753 fires)
  2018:  4,331,090 rows  (1,257 fires)
  2019:  4,331,090 rows  (517 fires)
  2020:  4,342,956 rows  (590 fires)
  2021:  4,331,090 rows  (1,113 fires)
  2022:  4,331,090 rows  (269 fires)

  Total rows (fire season): 33,936,760
  Total fire pixels       : 10,077

Extracting unique pixel locations...
  Unique pixels: 11,866

Aggregating fire events per pixel...
  Pixels with at least one fire: 4,250

  Fire count distribution:
    Max fires in one pixel: 80
    Mean fires per fire pixel: 2.37

  Dominant cause distribution:
    Undetermined             : 34 pixels
    Human                    : 1,231 pixels
    Natural (lightning)      : 2,981 pixels
  